In [ ]:
import sys

print("Python version:")
print(sys.version)

assert sys.version_info[:2] == (3, 11), "Please use Python 3.11"

print("\n✅ Python 3.11 is active.")

In [ ]:
from pathlib import Path

DATASET_DIR = Path(r"C:\Users\MURALIKRISHNA\Downloads\ShrimpDiseaseImageBD")

print("Dataset exists:", DATASET_DIR.exists())
print("Dataset path:", DATASET_DIR)

In [ ]:
ROOT_DIR = DATASET_DIR / "Root"

print("Root exists:", ROOT_DIR.exists())

if ROOT_DIR.exists():
    print("\nContents of Root:")
    for item in ROOT_DIR.iterdir():
        print(" -", item.name)

In [ ]:
from collections import Counter

RAW_DIR = ROOT_DIR / "Raw Images" / "Raw Images"

categories = {
    "Healthy": RAW_DIR / "1. Healthy",
    "BG": RAW_DIR / "2. BG",
    "WSSV": RAW_DIR / "3. WSSV",
    "WSSV_BG": RAW_DIR / "4. WSSV_BG",
}

image_extensions = {".jpg", ".jpeg", ".png"}

counts = {}

for category, folder in categories.items():
    images = [
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in image_extensions
    ]
    
    counts[category] = len(images)

print("RAW IMAGE COUNTS")
print("=" * 30)

for category, count in counts.items():
    print(f"{category:10s}: {count}")

print("=" * 30)
print(f"TOTAL      : {sum(counts.values())}")

In [ ]:
from PIL import Image

dimension_counts = Counter()

for category, folder in categories.items():
    for image_path in folder.rglob("*"):
        if image_path.is_file() and image_path.suffix.lower() in image_extensions:
            try:
                with Image.open(image_path) as img:
                    dimension_counts[img.size] += 1
            except Exception as e:
                print("Could not read:", image_path)
                print("Reason:", e)

print("IMAGE DIMENSIONS")
print("=" * 40)

for dimensions, count in dimension_counts.most_common():
    print(f"{dimensions}: {count} images")

In [ ]:
ANNOTATED_DIR = (
    ROOT_DIR
    / "Annotated Diseased Shrimp Images"
    / "Annotated Diseased Shrimp Images"
)

annotation_categories = {
    "BG": ANNOTATED_DIR / "1. BG",
    "WSSV": ANNOTATED_DIR / "2. WSSV",
    "WSSV_BG": ANNOTATED_DIR / "4. WSSV_BG",
}

print("ANNOTATED DATASET")
print("=" * 40)

for category, folder in annotation_categories.items():
    image_dir = folder / "images"
    label_dir = folder / "labels"

    image_count = len([
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ])

    label_count = len([
        p for p in label_dir.iterdir()
        if p.is_file() and p.suffix.lower() == ".txt"
    ])

    print(f"{category:10s} | images: {image_count:4d} | labels: {label_count:4d}")

In [ ]:
sample_labels = []

for category, folder in annotation_categories.items():
    label_dir = folder / "labels"

    for label_path in label_dir.glob("*.txt"):
        with open(label_path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]

        if lines:
            sample_labels.append(
                (category, label_path.name, lines[:5])
            )

        if len(sample_labels) >= 10:
            break

    if len(sample_labels) >= 10:
        break


print("SAMPLE YOLO ANNOTATIONS")
print("=" * 70)

for category, filename, lines in sample_labels:
    print(f"\nCategory: {category}")
    print(f"File    : {filename}")
    
    for line in lines:
        print(" ", line)

In [ ]:
annotation_stats = {}

for category, folder in annotation_categories.items():
    label_dir = folder / "labels"

    total_boxes = 0
    images_with_labels = 0

    for label_path in label_dir.glob("*.txt"):
        with open(label_path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]

        if lines:
            images_with_labels += 1
            total_boxes += len(lines)

    annotation_stats[category] = {
        "images_with_labels": images_with_labels,
        "total_boxes": total_boxes,
        "average_boxes_per_image": (
            total_boxes / images_with_labels
            if images_with_labels else 0
        )
    }

print("ANNOTATION STATISTICS")
print("=" * 65)

for category, stats in annotation_stats.items():
    print(
        f"{category:10s} | "
        f"images: {stats['images_with_labels']:3d} | "
        f"boxes: {stats['total_boxes']:5d} | "
        f"avg/image: {stats['average_boxes_per_image']:.2f}"
    )

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import random

def show_annotation(category, image_name=None):
    """
    Display an annotated shrimp image together with its
    existing YOLO disease-region bounding boxes.
    """

    folder = annotation_categories[category]
    image_dir = folder / "images"
    label_dir = folder / "labels"

    # Choose an image if one wasn't specified
    if image_name is None:
        image_paths = [
            p for p in image_dir.iterdir()
            if p.is_file() and p.suffix.lower() in image_extensions
        ]
        image_path = random.choice(image_paths)
    else:
        image_path = image_dir / image_name

    label_path = label_dir / f"{image_path.stem}.txt"

    # Open image
    image = Image.open(image_path).convert("RGB")
    width, height = image.size

    # Draw annotations
    draw = ImageDraw.Draw(image)

    if label_path.exists():
        with open(label_path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]

        for line in lines:
            parts = line.split()

            if len(parts) != 5:
                continue

            class_id, xc, yc, bw, bh = map(float, parts)

            # Convert normalized YOLO coordinates to pixels
            x_center = xc * width
            y_center = yc * height
            box_width = bw * width
            box_height = bh * height

            x1 = int(x_center - box_width / 2)
            y1 = int(y_center - box_height / 2)
            x2 = int(x_center + box_width / 2)
            y2 = int(y_center + box_height / 2)

            draw.rectangle(
                [x1, y1, x2, y2],
                outline="red",
                width=4
            )

    # Display
    plt.figure(figsize=(9, 9))
    plt.imshow(image)
    plt.title(
        f"{category} | {image_path.name}\n"
        f"Existing annotations: disease-region boxes"
    )
    plt.axis("off")
    plt.show()

    print("Image :", image_path.name)
    print("Size  :", image.size)
    print("Label :", label_path)

In [ ]:
for category in ["BG", "WSSV", "WSSV_BG"]:
    show_annotation(category)

In [ ]:
def show_raw_samples():
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    axes = axes.ravel()

    for ax, (category, folder) in zip(axes, categories.items()):

        image_paths = [
            p for p in folder.iterdir()
            if p.is_file() and p.suffix.lower() in image_extensions
        ]

        image_path = random.choice(image_paths)

        image = Image.open(image_path).convert("RGB")

        ax.imshow(image)
        ax.set_title(category)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_raw_samples()

In [ ]:
# Test whole-shrimp box detection on one image from each category

test_images = {}

for category, folder in categories.items():
    image_paths = [
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ]

    test_images[category] = random.choice(image_paths)


for category, image_path in test_images.items():

    box, mask = find_shrimp_box(image_path)

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(8, 8))
    plt.imshow(image)

    if box is not None:
        x1, y1, x2, y2 = box

        rect = plt.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            linewidth=3
        )

        plt.gca().add_patch(rect)

    plt.title(f"{category} - Candidate Whole-Shrimp Box")
    plt.axis("off")
    plt.show()

    print(f"{category}: {image_path.name}")
    print("Candidate box:", box)

In [ ]:
%pip install opencv-python

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def find_shrimp_box(image_path):
    """
    Estimate the whole-shrimp bounding box from a single-shrimp image.

    This is a CANDIDATE box only.
    We will visually verify it before creating the final YOLO labels.
    """

    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    h, w = image.shape[:2]

    # Convert to HSV
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # ---------------------------------------------------------
    # Create a foreground mask.
    # Shrimp tends to differ from the blue background mainly
    # in saturation/value/color distribution.
    # ---------------------------------------------------------

    # Blue-background mask
    blue_mask = cv2.inRange(
        hsv,
        np.array([80, 40, 40]),
        np.array([140, 255, 255])
    )

    # White/light-background mask
    white_mask = cv2.inRange(
        hsv,
        np.array([0, 0, 150]),
        np.array([180, 80, 255])
    )

    # Combine likely background regions
    background = cv2.bitwise_or(blue_mask, white_mask)

    # Foreground = inverse background
    foreground = cv2.bitwise_not(background)

    # Remove small noise
    kernel = np.ones((7, 7), np.uint8)

    foreground = cv2.morphologyEx(
        foreground,
        cv2.MORPH_OPEN,
        kernel,
        iterations=2
    )

    foreground = cv2.morphologyEx(
        foreground,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=4
    )

    # ---------------------------------------------------------
    # Find contours
    # ---------------------------------------------------------

    contours, _ = cv2.findContours(
        foreground,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return None, foreground

    # Ignore very tiny regions
    min_area = 0.002 * w * h

    valid_contours = [
        c for c in contours
        if cv2.contourArea(c) >= min_area
    ]

    if not valid_contours:
        return None, foreground

    # Use the largest meaningful foreground region
    contour = max(valid_contours, key=cv2.contourArea)

    x, y, bw, bh = cv2.boundingRect(contour)

    # ---------------------------------------------------------
    # Slightly expand the box to include antennae/legs/tail.
    # ---------------------------------------------------------

    padding_x = int(bw * 0.08)
    padding_y = int(bh * 0.08)

    x1 = max(0, x - padding_x)
    y1 = max(0, y - padding_y)
    x2 = min(w, x + bw + padding_x)
    y2 = min(h, y + bh + padding_y)

    return (x1, y1, x2, y2), foreground

In [ ]:
# Pick one random image from each category
test_images = {}

for category, folder in categories.items():
    image_paths = [
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ]

    test_images[category] = random.choice(image_paths)


for category, image_path in test_images.items():

    box, mask = find_shrimp_box(image_path)

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(8, 8))
    plt.imshow(image)

    if box is not None:
        x1, y1, x2, y2 = box

        rectangle = plt.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            linewidth=3
        )

        plt.gca().add_patch(rectangle)

    plt.title(f"{category} — Candidate shrimp box")
    plt.axis("off")
    plt.show()

    print(category)
    print("Image:", image_path.name)
    print("Box:", box)

In [ ]:
# 
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import random

print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)

print("\n✅ Required libraries are ready.")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random

# ---------------------------------------------------------
# STEP 2: TEST AUTOMATIC WHOLE-SHRIMP BOX GENERATION
# ---------------------------------------------------------

random.seed(42)

# Collect image paths from all 4 categories
sample_images = []

for category, folder in categories.items():
    image_paths = [
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ]
    
    selected = random.sample(image_paths, min(5, len(image_paths)))
    
    for image_path in selected:
        sample_images.append((category, image_path))

print(f"Testing {len(sample_images)} images")
print("=" * 60)


def detect_shrimp_box(image_path):
    """
    Generate a candidate bounding box around the complete shrimp.
    This is only a TESTING algorithm.
    It does not modify the original image.
    """

    image = cv2.imread(str(image_path))

    if image is None:
        return None, None

    # OpenCV uses BGR
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # -----------------------------------------------------
    # Blue background detection
    # -----------------------------------------------------
    # Blue background generally has:
    # H roughly 80-130
    # S reasonably high
    # V reasonably high
    # These values are intentionally conservative.
    lower_blue = np.array([80, 50, 40])
    upper_blue = np.array([135, 255, 255])

    blue_mask = cv2.inRange(hsv, lower_blue, upper_blue)

    # Foreground = areas that are NOT blue
    foreground = cv2.bitwise_not(blue_mask)

    # -----------------------------------------------------
    # Clean the foreground mask
    # -----------------------------------------------------
    kernel = np.ones((7, 7), np.uint8)

    foreground = cv2.morphologyEx(
        foreground,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=2
    )

    foreground = cv2.morphologyEx(
        foreground,
        cv2.MORPH_OPEN,
        kernel,
        iterations=1
    )

    # -----------------------------------------------------
    # Find connected components
    # -----------------------------------------------------
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        foreground,
        connectivity=8
    )

    if num_labels <= 1:
        return image, None

    # Image dimensions
    h, w = image.shape[:2]
    image_area = h * w

    candidates = []

    for i in range(1, num_labels):

        x = stats[i, cv2.CC_STAT_LEFT]
        y = stats[i, cv2.CC_STAT_TOP]
        bw = stats[i, cv2.CC_STAT_WIDTH]
        bh = stats[i, cv2.CC_STAT_HEIGHT]
        area = stats[i, cv2.CC_STAT_AREA]

        box_area = bw * bh

        # Ignore tiny noise
        if area < image_area * 0.001:
            continue

        # Ignore components occupying almost the whole image
        if box_area > image_area * 0.85:
            continue

        candidates.append(
            (area, x, y, bw, bh)
        )

    if not candidates:
        return image, None

    # Largest meaningful foreground component
    candidates.sort(reverse=True)

    _, x, y, bw, bh = candidates[0]

    # -----------------------------------------------------
    # Add a small margin around shrimp
    # -----------------------------------------------------
    margin = 15

    x1 = max(0, x - margin)
    y1 = max(0, y - margin)

    x2 = min(w - 1, x + bw + margin)
    y2 = min(h - 1, y + bh + margin)

    return image, (x1, y1, x2, y2)


# ---------------------------------------------------------
# Display results
# ---------------------------------------------------------

fig, axes = plt.subplots(5, 4, figsize=(16, 20))
axes = axes.flatten()

for ax, (category, image_path) in zip(axes, sample_images):

    image, box = detect_shrimp_box(image_path)

    if image is None:
        ax.set_title("Could not read image")
        ax.axis("off")
        continue

    # Convert BGR -> RGB
    display_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    ax.imshow(display_image)

    if box is not None:

        x1, y1, x2, y2 = box

        rectangle = plt.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            linewidth=2,
            edgecolor="red"
        )

        ax.add_patch(rectangle)

        ax.set_title(
            f"{category}\n"
            f"Box: ({x1}, {y1}) → ({x2}, {y2})"
        )

    else:
        ax.set_title(
            f"{category}\n❌ No shrimp box detected"
        )

    ax.axis("off")


plt.tight_layout()
plt.show()

print("\n✅ Step 2 completed.")
print("These are ONLY candidate boxes.")
print("Original dataset has NOT been modified.")